# DPO-SDXL × DrawBench — Evaluation (Colab)

Compares Standard Sampling vs Z-Sampling on DPO-finetuned SDXL across
all 200 DrawBench prompts × 3 seeds.

**Metrics** (from the paper's Table 1, SDXL + DrawBench column):
- HPS v2 ↑
- AES (Aesthetic Score) ↑
- PickScore ↑
- IR (ImageReward) ↑

**Dataset**: 1200 images total (600 DPO standard + 600 DPO z-sampling),
stored as checkpoint zips on Drive from the generation notebooks.

Requires GPU runtime (T4 is sufficient for evaluation).

In [1]:
!pip install -q transformers accelerate torch torchvision pillow pandas
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q git+https://github.com/THUDM/ImageReward.git
!pip install -q hpsv2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     / 4.3 MB 6.5 MB/s 0:00:000mm
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 22.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 95.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.8/316.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are 

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 1. Data Preparation

Copy the two dataset zips from Drive to local disk

In [4]:
DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling")

DATA_DIR = Path("/content/data")
STANDARD_DIR = DATA_DIR / "dpo_sdxl_drawbench"
ZSAMPLING_DIR = DATA_DIR / "dpo_sdxl_drawbench_zsampling"

SEEDS = [42, 123, 2026]
NUM_PROMPTS = 200

In [5]:
import shutil
import time

DATA_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
for name in ["dpo_sdxl_drawbench.zip", "dpo_sdxl_drawbench_zsampling.zip"]:
    src = DRIVE_DIR / name
    dst = DATA_DIR / name
    if not dst.exists():
        print(f"Copying {name} ({src.stat().st_size / 1e6:.0f} MB)...")
        shutil.copy2(src, dst)
    else:
        print(f"{name} already copied.")

for name in ["dpo_sdxl_drawbench.zip", "dpo_sdxl_drawbench_zsampling.zip"]:
    target = DATA_DIR / name.replace(".zip", "")
    if not target.exists():
        print(f"Extracting {name}...")
        with zipfile.ZipFile(DATA_DIR / name) as z:
            z.extractall(DATA_DIR)
    else:
        print(f"{name.replace('.zip', '')}/ already extracted.")

print(f"\nDone in {time.time() - t0:.1f}s")
print(f"Standard: {STANDARD_DIR}")
print(f"Z-Sampling: {ZSAMPLING_DIR}")

Copying dpo_sdxl_drawbench.zip (924 MB)...
Copying dpo_sdxl_drawbench_zsampling.zip (702 MB)...
Extracting dpo_sdxl_drawbench.zip...
Extracting dpo_sdxl_drawbench_zsampling.zip...

Done in 57.7s
Standard: /content/data/dpo_sdxl_drawbench
Z-Sampling: /content/data/dpo_sdxl_drawbench_zsampling


In [6]:
def index_images(base_dir: Path) -> dict[str, Path]:
    """Build a filename -> path index across all checkpoint subfolders."""
    index = {}
    for png in base_dir.rglob("*.png"):
        index[png.name] = png
    return index

std_index = index_images(STANDARD_DIR)
zs_index = index_images(ZSAMPLING_DIR)

print(f"Standard images found: {len(std_index)} (expected 600)")
print(f"Z-Sampling images found: {len(zs_index)} (expected 600)")
assert len(std_index) == 600, f"Expected 600 standard images, got {len(std_index)}"
assert len(zs_index) == 600, f"Expected 600 z-sampling images, got {len(zs_index)}"

Standard images found: 600 (expected 600)
Z-Sampling images found: 600 (expected 600)


## 2. Load DrawBench Prompts

Same source as the generation notebooks — ensures prompt index alignment.

In [7]:
CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df_prompts = pd.read_csv(CSV_URL)
PROMPTS = df_prompts["Prompts"].tolist()
assert len(PROMPTS) == NUM_PROMPTS, f"Expected {NUM_PROMPTS} prompts, got {len(PROMPTS)}"
print(f"Loaded {len(PROMPTS)} DrawBench prompts")
print(f"Examples: {PROMPTS[:3]}")

Loaded 200 DrawBench prompts
Examples: ['A red colored car.', 'A black colored car.', 'A pink colored car.']


In [8]:
def build_pairs() -> pd.DataFrame:
    """Build a DataFrame of (prompt_idx, seed, prompt, standard_path, zsampling_path)."""
    rows = []
    missing = []
    for seed in SEEDS:
        for idx in range(NUM_PROMPTS):
            std_name = f"dpo_sdxl_prompt{idx:03d}_seed{seed}.png"
            zs_name = f"dpo_sdxl_zsample_prompt{idx:03d}_seed{seed}.png"
            std_path = std_index.get(std_name)
            zs_path = zs_index.get(zs_name)
            if std_path is None or zs_path is None:
                missing.append((idx, seed, std_name if std_path is None else zs_name))
                continue
            rows.append({
                "prompt_idx": idx,
                "seed": seed,
                "prompt": PROMPTS[idx],
                "standard_path": str(std_path),
                "zsampling_path": str(zs_path),
            })
    if missing:
        print(f"WARNING: {len(missing)} missing images!")
        for idx, seed, name in missing[:5]:
            print(f"  prompt={idx} seed={seed}: {name}")
    else:
        print("All 600 pairs verified.")
    return pd.DataFrame(rows)

df_pairs = build_pairs()
print(f"Total pairs: {len(df_pairs)}")
df_pairs.head()

All 600 pairs verified.
Total pairs: 600


,prompt_idx,seed,prompt,standard_path,zsampling_path
0,0,42,A red colored car.,/content/data/dpo_sdxl_drawbench/dpo_sdxl_draw...,/content/data/dpo_sdxl_drawbench_zsampling/dpo...
1,1,42,A black colored car.,/content/data/dpo_sdxl_drawbench/dpo_sdxl_draw...,/content/data/dpo_sdxl_drawbench_zsampling/dpo...
2,2,42,A pink colored car.,/content/data/dpo_sdxl_drawbench/dpo_sdxl_draw...,/content/data/dpo_sdxl_drawbench_zsampling/dpo...
3,3,42,A black colored dog.,/content/data/dpo_sdxl_drawbench/dpo_sdxl_draw...,/content/data/dpo_sdxl_drawbench_zsampling/dpo...
4,4,42,A red colored dog.,/content/data/dpo_sdxl_drawbench/dpo_sdxl_draw...,/content/data/dpo_sdxl_drawbench_zsampling/dpo...


## 3. Scoring

### 3.1 HPS v2

In [9]:
import hpsv2
import pathlib, shutil, urllib.request

_bpe_dst = pathlib.Path(hpsv2.__file__).parent / "src" / "open_clip" / "bpe_simple_vocab_16e6.txt.gz"
if not _bpe_dst.exists():
    _bpe_src = None
    for _mod in ("clip", "open_clip"):
        try:
            _m = __import__(_mod)
            _cand = pathlib.Path(_m.__file__).parent / "bpe_simple_vocab_16e6.txt.gz"
            if _cand.exists():
                _bpe_src = _cand
                break
        except Exception:
            pass
    _bpe_dst.parent.mkdir(parents=True, exist_ok=True)
    if _bpe_src is not None:
        shutil.copy(_bpe_src, _bpe_dst)
    else:
        _url = "https://github.com/tgxs002/HPSv2/raw/master/src/open_clip/bpe_simple_vocab_16e6.txt.gz"
        urllib.request.urlretrieve(_url, _bpe_dst)
    print(f"Installed BPE vocab -> {_bpe_dst}")

def score_hpsv2(image_paths: list[str], prompt: str) -> list[float]:
    raw = hpsv2.score(image_paths, prompt, hps_version="v2.1")
    return np.asarray(raw, dtype=float).reshape(-1).tolist()

Installed BPE vocab -> /usr/local/lib/python3.12/dist-packages/hpsv2/src/open_clip/bpe_simple_vocab_16e6.txt.gz


### 3.2 Aesthetic Score (AES)

In [10]:
import clip

class _AesMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(768, 1024),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(1024, 128),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(128, 64),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(64, 16),
            torch.nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

class AestheticScorer:
    def __init__(self, device=DEVICE):
        self.device = device
        self.clip_model, self.preprocess = clip.load("ViT-L/14", device=device)
        url = "https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac+logos+ava1-l14-linearMSE.pth"
        state_dict = torch.hub.load_state_dict_from_url(url, map_location=device)
        self.mlp = _AesMLP()
        self.mlp.load_state_dict(state_dict)
        self.mlp.to(device).eval()

    @torch.no_grad()
    def score(self, image_paths: list[str]) -> list[float]:
        scores = []
        for path in image_paths:
            img = self.preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(self.device)
            emb = self.clip_model.encode_image(img).float()
            emb = emb / emb.norm(dim=-1, keepdim=True)
            score = self.mlp(emb).item()
            scores.append(score)
        return scores

aes_scorer = AestheticScorer()
print("AES scorer loaded.")

100%|████████████████████████████████████████| 890M/890M [00:06<00:00, 148MiB/s]


Downloading: "https://github.com/christophschuhmann/improved-aesthetic-predictor/raw/main/sac+logos+ava1-l14-linearMSE.pth" to /root/.cache/torch/hub/checkpoints/sac+logos+ava1-l14-linearMSE.pth


100%|██████████| 3.54M/3.54M [00:00<00:00, 350MB/s]

AES scorer loaded.


### 3.3 PickScore

In [11]:
from transformers import AutoProcessor, AutoModel

def _pooled(out):
    return out if isinstance(out, torch.Tensor) else out.pooler_output

class PickScorer:
    def __init__(self, device=DEVICE):
        self.device = device
        self.processor = AutoProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
        self.model = AutoModel.from_pretrained("yuvalkirstain/PickScore_v1").eval().to(device)

    @torch.no_grad()
    def score(self, image_paths: list[str], prompt: str) -> list[float]:
        scores = []
        text_inputs = self.processor(
            text=prompt, padding=True, truncation=True,
            max_length=77, return_tensors="pt"
        ).to(self.device)
        text_emb = _pooled(self.model.get_text_features(**text_inputs))
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

        for path in image_paths:
            img = Image.open(path).convert("RGB")
            img_inputs = self.processor(
                images=img, return_tensors="pt"
            ).to(self.device)
            img_emb = _pooled(self.model.get_image_features(**img_inputs))
            img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
            score = (self.model.logit_scale.exp() * (text_emb @ img_emb.T)).item()
            scores.append(score)
        return scores

pick_scorer = PickScorer()
print("PickScore scorer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.51k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/910 [00:00<?, ?it/s]

PickScore scorer loaded.


### 3.4 ImageReward (IR)

In [12]:
import math

SKIP_IMAGEREWARD = True

class IRScorer:
    """Placeholder ImageReward scorer that returns NaN while IR is skipped."""
    def __init__(self, device=DEVICE):
        self.enabled = not SKIP_IMAGEREWARD

    def score(self, image_paths: list[str], prompt: str) -> list[float]:
        return [float("nan") for _ in image_paths]

ir_scorer = IRScorer()
print("ImageReward scorer SKIPPED (returning NaN).")

ImageReward scorer SKIPPED (returning NaN).


## 4. Run Evaluation

Score all 1200 images across the 4 metrics. Results are saved incrementally.

In [13]:
import json
import time
import math
from tqdm.auto import tqdm

RESULTS_PATH = DRIVE_DIR / "dpo_sdxl_drawbench" / "dpo_sdxl_drawbench_eval_results.jsonl"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS = ["hps", "aes", "pick", "ir"]

def _fmt_metric(name, s_std, s_zs):
    if math.isnan(s_std) or math.isnan(s_zs):
        return f"{name}=nan"
    win = "Z+" if s_zs > s_std else ("S+" if s_std > s_zs else "==")
    return f"{name} {s_std:.3f}->{s_zs:.3f} {win}"

def score_all_pairs(df: pd.DataFrame, resume: bool = True, verbose: bool = True) -> pd.DataFrame:
    """Score all pairs and save results incrementally to JSONL, with live progress."""
    scored = []

    if resume and RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            scored = [json.loads(line) for line in f if line.strip()]
        done_keys = {(r["prompt_idx"], r["seed"]) for r in scored}
        print(f"Resuming: {len(scored)} already scored")
    else:
        done_keys = set()
        RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

    pending = df[~df.apply(lambda r: (r["prompt_idx"], r["seed"]) in done_keys, axis=1)]
    print(f"Scoring {len(pending)} remaining pairs...")

    wins = {m: 0 for m in METRICS}
    counted = {m: 0 for m in METRICS}
    t0 = time.time()
    bar = tqdm(pending.iterrows(), total=len(pending), desc="Scoring", unit="pair")
    for i, (_, row) in enumerate(bar):
        paths = [row["standard_path"], row["zsampling_path"]]
        prompt = row["prompt"]

        scores = {
            "hps": score_hpsv2(paths, prompt),
            "aes": aes_scorer.score(paths),
            "pick": pick_scorer.score(paths, prompt),
            "ir": ir_scorer.score(paths, prompt),
        }

        result = {
            "prompt_idx": int(row["prompt_idx"]),
            "seed": int(row["seed"]),
            "prompt": prompt,
        }
        for m in METRICS:
            s_std, s_zs = float(scores[m][0]), float(scores[m][1])
            result[f"{m}_standard"] = s_std
            result[f"{m}_zsampling"] = s_zs
            if not (math.isnan(s_std) or math.isnan(s_zs)):
                counted[m] += 1
                if s_zs > s_std:
                    wins[m] += 1
        scored.append(result)

        with open(RESULTS_PATH, "a") as f:
            f.write(json.dumps(result) + "\n")

        bar.set_postfix({m: f"{wins[m]/counted[m]*100:.0f}%" for m in METRICS if counted[m]})

        if verbose:
            detail = " | ".join(_fmt_metric(m.upper(), scores[m][0], scores[m][1]) for m in METRICS)
            snippet = (prompt[:40] + "...") if len(prompt) > 40 else prompt
            tqdm.write(f"[{i+1:>4}/{len(pending)}] seed{int(row['seed'])} | {detail} | \"{snippet}\"")

    elapsed = time.time() - t0
    print(f"\nDone. Scored {len(pending)} pairs in {elapsed/60:.1f} min")
    print("Z-Sampling win-rate (excl. NaN): " + ", ".join(
        f"{m.upper()} {wins[m]/counted[m]*100:.1f}%" if counted[m] else f"{m.upper()} n/a" for m in METRICS))
    return pd.DataFrame(scored)

df_results = score_all_pairs(df_pairs)

Scoring 600 remaining pairs...


Scoring:   0%|          | 0/600 [00:00<?, ?pair/s]

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


open_clip_pytorch_model.bin:   0%|          | 0.00/3.94G [00:00<?, ?B/s]

HPS_v2.1_compressed.pt:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/hpsv2/img_score.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


[   1/600] seed42 | HPS 0.261->0.287 Z+ | AES 5.430->6.167 Z+ | PICK 23.020->23.239 Z+ | IR=nan | "A red colored car."
[   2/600] seed42 | HPS 0.242->0.276 Z+ | AES 5.309->5.667 Z+ | PICK 22.791->23.106 Z+ | IR=nan | "A black colored car."
[   3/600] seed42 | HPS 0.293->0.310 Z+ | AES 5.974->6.290 Z+ | PICK 24.490->24.806 Z+ | IR=nan | "A pink colored car."
[   4/600] seed42 | HPS 0.297->0.306 Z+ | AES 6.178->5.564 S+ | PICK 23.566->23.046 S+ | IR=nan | "A black colored dog."
[   5/600] seed42 | HPS 0.290->0.299 Z+ | AES 5.883->6.167 Z+ | PICK 23.189->23.075 S+ | IR=nan | "A red colored dog."
[   6/600] seed42 | HPS 0.314->0.313 S+ | AES 6.195->6.403 Z+ | PICK 23.768->23.473 S+ | IR=nan | "A blue colored dog."
[   7/600] seed42 | HPS 0.260->0.268 Z+ | AES 5.486->5.311 S+ | PICK 22.412->22.551 Z+ | IR=nan | "A green colored banana."
[   8/600] seed42 | HPS 0.267->0.271 Z+ | AES 5.569->5.705 Z+ | PICK 22.976->22.785 S+ | IR=nan | "A red colored banana."
[   9/600] seed42 | HPS 0.268->0.2

## 5. Results

In [14]:
print(f"Total scored pairs: {len(df_results)}")
df_results.head()

Total scored pairs: 600


,prompt_idx,seed,prompt,hps_standard,hps_zsampling,aes_standard,aes_zsampling,pick_standard,pick_zsampling,ir_standard,ir_zsampling
0,0,42,A red colored car.,0.261475,0.287109,5.429587,6.167072,23.019945,23.239222,NaN,NaN
1,1,42,A black colored car.,0.241577,0.275635,5.309115,5.667014,22.790588,23.105598,NaN,NaN
2,2,42,A pink colored car.,0.292725,0.309570,5.974072,6.290138,24.490145,24.806398,NaN,NaN
3,3,42,A black colored dog.,0.297363,0.306152,6.178196,5.563796,23.565847,23.046303,NaN,NaN
4,4,42,A red colored dog.,0.290283,0.299316,5.882805,6.167044,23.189228,23.074610,NaN,NaN


In [15]:
summary_rows = []
for m in METRICS:
    std_col = f"{m}_standard"
    zs_col = f"{m}_zsampling"
    valid = df_results[[std_col, zs_col]].dropna()
    if len(valid) == 0:
        continue
    summary_rows.append({
        "metric": m.upper(),
        "standard_mean": valid[std_col].mean(),
        "zsampling_mean": valid[zs_col].mean(),
        "delta": valid[zs_col].mean() - valid[std_col].mean(),
        "zsampling_win_rate": (valid[zs_col] > valid[std_col]).mean(),
        "n_valid": len(valid),
    })

df_summary = pd.DataFrame(summary_rows)
print("\n=== DPO-SDXL DrawBench: Standard vs Z-Sampling ===")
print(df_summary.to_string(index=False))


=== DPO-SDXL DrawBench: Standard vs Z-Sampling ===
metric  standard_mean  zsampling_mean     delta  zsampling_win_rate  n_valid
   HPS       0.281462        0.289628  0.008167            0.658333      600
   AES       5.535490        5.638801  0.103311            0.650000      600
  PICK      22.612367       22.503907 -0.108460            0.396667      600


In [16]:
# Save results to Drive
csv_path = RESULTS_PATH.parent / "dpo_sdxl_drawbench_eval_results.csv"
df_results.to_csv(csv_path, index=False)

summary_csv = RESULTS_PATH.parent / "dpo_sdxl_drawbench_eval_summary.csv"
df_summary.to_csv(summary_csv, index=False)

os.sync()
print(f"Results saved to {csv_path}")
print(f"Summary saved to {summary_csv}")

Results saved to /content/drive/MyDrive/generative-modelling/dpo_sdxl_drawbench/dpo_sdxl_drawbench_eval_results.csv
Summary saved to /content/drive/MyDrive/generative-modelling/dpo_sdxl_drawbench/dpo_sdxl_drawbench_eval_summary.csv
